In [9]:
import contextlib
import importlib.util
import io
import sys
from pathlib import Path

import mpmath as mp
import numpy as np


def _find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "covariant formalism/python2").is_dir():
            return base
    raise RuntimeError("Could not find the project root containing covariant formalism/python2.")


ROOT = _find_project_root()
PY2_DIR = ROOT / "covariant formalism/python2"


def _load_python2_module(alias, filename):
    spec = importlib.util.spec_from_file_location(alias, PY2_DIR / filename)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {filename} from {PY2_DIR}.")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


@contextlib.contextmanager
def _temporary_sys_modules(mapping):
    saved = {name: sys.modules.get(name) for name in mapping}
    try:
        for name, module in mapping.items():
            sys.modules[name] = module
        yield
    finally:
        for name, module in saved.items():
            if module is None:
                sys.modules.pop(name, None)
            else:
                sys.modules[name] = module


PY2_RGG = _load_python2_module("python2_ribbon_graph_generator", "ribbon_graph_generator.py")
PY2_PF = _load_python2_module("python2_partition_function", "partition_function.py")
with _temporary_sys_modules(
    {
        "partition_function": PY2_PF,
        "ribbon_graph_generator": PY2_RGG,
    }
):
    PY2_ET = _load_python2_module("python2_ell_to_tau", "ell_to_tau.py")


@contextlib.contextmanager
def _python2_import_context():
    with _temporary_sys_modules(
        {
            "partition_function": PY2_PF,
            "ribbon_graph_generator": PY2_RGG,
            "ell_to_tau": PY2_ET,
        }
    ):
        yield


def _genus1_noncompact_factor(ribbon_graph, ell_list):
    forms = PY2_ET.make_cyl_eqn_improved_higher_genus(ribbon_graph, ell_list)
    pdata = PY2_ET.period_matrix(
        forms=forms,
        ribbon_graph=ribbon_graph,
        ell_list=ell_list,
        return_data=True,
    )
    omega = np.atleast_2d(np.asarray(pdata["Omega"], dtype=np.complex128))
    A_periods = np.atleast_2d(np.asarray(pdata["A_periods"], dtype=np.complex128))
    tau = complex(omega[0, 0])
    P1 = complex(A_periods[0, 0])
    avg_nu = np.ravel(np.asarray(
        PY2_ET.average_nu(forms=forms, ribbon_graph=ribbon_graph, ell_list=ell_list),
        dtype=np.complex128,
    ))
    eta_abs = abs(PY2_ET.dedekind_eta(tau))
    ghost_factor = (abs(P1) ** (-2.0)) * (abs(avg_nu[0]) ** (11.0 / 6.0)) * (eta_abs ** 4.0)
    matter_factor = (tau.imag ** (-13.0)) * (eta_abs ** (-52.0))
    factor = ghost_factor * matter_factor
    return {
        "forms": forms,
        "Omega": omega,
        "A_periods": A_periods,
        "P1": P1,
        "tau": tau,
        "avg_nu": avg_nu,
        "ghost_factor": float(ghost_factor),
        "matter_factor": float(matter_factor),
        "factor": float(factor),
    }


with _python2_import_context():
    with contextlib.redirect_stdout(io.StringIO()):
        genus1_graphs = PY2_RGG.generate_ribbon_graphs(1, 3)
    assert len(genus1_graphs) == 1
    genus1_graph = genus1_graphs[0]

    genus1_pair_data = [
        (
            "scale 12",
            [12 * x for x in [5, 7, 1]],
            [12 * x for x in [3, 1, 9]],
        ),
         (
            "scale 24",
            [24 * x for x in [5, 7, 1]],
            [24 * x for x in [3, 1, 9]],
        )
    ]

    print("genus 1 non-compact ratios from four total ell sets")
    for label, genus1_ref, genus1_num in genus1_pair_data:
        assert sum(genus1_ref) == sum(genus1_num)
        genus1_log_b_ref = PY2_PF.traced_bdet_log_f1(genus1_graph, genus1_ref)
        genus1_log_b_num = PY2_PF.traced_bdet_log_f1(genus1_graph, genus1_num)
        genus1_log_A_ref = PY2_PF.traced_prime_det_log_f1(genus1_graph, genus1_ref)
        genus1_log_A_num = PY2_PF.traced_prime_det_log_f1(genus1_graph, genus1_num)
        genus1_numeric_ratio = complex(mp.e ** (0.5 * (genus1_log_b_num - genus1_log_b_ref) - 13.0 * (genus1_log_A_num - genus1_log_A_ref)))

        genus1_data_ref = _genus1_noncompact_factor(genus1_graph, genus1_ref)
        genus1_data_num = _genus1_noncompact_factor(genus1_graph, genus1_num)
        assert genus1_data_ref["Omega"].shape == (1, 1)
        assert genus1_data_num["Omega"].shape == (1, 1)
        assert genus1_data_ref["tau"].imag > 0.0
        assert genus1_data_num["tau"].imag > 0.0

        genus1_analytic_ratio = genus1_data_num["factor"] / genus1_data_ref["factor"]
        genus1_rel = abs(genus1_numeric_ratio - genus1_analytic_ratio) / abs(genus1_analytic_ratio)

        print(label)
        print("  ref ell   =", genus1_ref)
        print("  num ell   =", genus1_num)
        print("  numeric   =", genus1_numeric_ratio)
        print("  analytic  =", genus1_analytic_ratio)
        print("  ghost ref =", genus1_data_ref["ghost_factor"])
        print("  ghost num =", genus1_data_num["ghost_factor"])
        print("  matter ref=", genus1_data_ref["matter_factor"])
        print("  matter num=", genus1_data_num["matter_factor"])
        print("  rel diff  =", genus1_rel)

    with contextlib.redirect_stdout(io.StringIO()):
        genus2_graphs = PY2_RGG.generate_ribbon_graphs(1, 9)
    assert len(genus2_graphs) == 9
    genus2_graph = genus2_graphs[1]

    genus2_scale = 4
    genus2_bases = [
        [12, 10, 11, 11, 12, 10, 11, 11, 11],
        [13, 9, 11, 11, 13, 9, 11, 11, 11],
        [14, 8, 11, 11, 14, 8, 11, 11, 11],
        [15, 7, 11, 11, 15, 7, 11, 11, 11],
    ]
    genus2_sets = [[genus2_scale * x for x in base] for base in genus2_bases]
    genus2_ref = genus2_sets[0]
    genus2_log_ref = PY2_PF.traced_numeric_amplitude_log_psi1_f1(genus2_graph, genus2_ref)
    genus2_forms_ref = PY2_ET.make_cyl_eqn_improved_higher_genus(genus2_graph, genus2_ref)
    genus2_data_ref = PY2_ET.genus2_matter_bc_candidate(
        genus2_forms_ref,
        ribbon_graph=genus2_graph,
        ell_list=genus2_ref,
    )
    genus2_omega_ref = np.asarray(genus2_data_ref["Omega"], dtype=np.complex128)
    assert genus2_omega_ref.shape == (2, 2)
    assert float(np.min(np.linalg.eigvalsh(np.imag(genus2_omega_ref)))) > 0.0

    print("genus 2 non-compact ratios from four total ell sets")
    print("systematic family: shift the first/second and fifth/sixth entries by the same amount")
    for idx, genus2_num in enumerate(genus2_sets[1:], start=1):
        assert sum(genus2_ref) == sum(genus2_num)
        genus2_log_num = PY2_PF.traced_numeric_amplitude_log_psi1_f1(genus2_graph, genus2_num)
        genus2_numeric_ratio = complex(mp.e ** (genus2_log_num - genus2_log_ref))

        genus2_forms_num = PY2_ET.make_cyl_eqn_improved_higher_genus(genus2_graph, genus2_num)
        genus2_data_num = PY2_ET.genus2_matter_bc_candidate(
            genus2_forms_num,
            ribbon_graph=genus2_graph,
            ell_list=genus2_num,
        )
        genus2_omega_num = np.asarray(genus2_data_num["Omega"], dtype=np.complex128)
        assert genus2_omega_num.shape == (2, 2)
        assert float(np.min(np.linalg.eigvalsh(np.imag(genus2_omega_num)))) > 0.0

        genus2_analytic_ratio = genus2_data_num["candidate_factor"] / genus2_data_ref["candidate_factor"]
        genus2_rel = abs(genus2_numeric_ratio - genus2_analytic_ratio) / abs(genus2_analytic_ratio)

        print(f"case {idx}")
        print("  ref ell   =", genus2_ref)
        print("  num ell   =", genus2_num)
        print("  numeric   =", genus2_numeric_ratio)
        print("  analytic  =", genus2_analytic_ratio)
        print("  rel diff  =", genus2_rel)


genus 1 non-compact ratios from four total ell sets
scale 12
  ref ell   = [60, 84, 12]
  num ell   = [36, 12, 108]
  numeric   = (3.6081977962838487+0j)
  analytic  = 4.660069842312191
  rel diff  = 0.22572023201833252


KeyboardInterrupt: 